# **BUILDING CLAUDE MANAGED AGENTS**

- Tutorial to learn how to build claude managed agents with Anthropic's Python SDK, cloud sandbox, tools, skills, file uploads, session managment, code execution, and automated Excel report generation.
- Managed clients are built around four main parts:
    1. **Agent:** the reusable configuration that contains the model, system prompt, tools, MCP servers, and skills.
    2. **Environment:** the location where the agent runs, either in Anthropic's cloud sandbox or in self-hosted sandbox.
    3. **Session:** a running instance of the agent that performs a specific task.
    4. **Events:** the messages, tool calls, results, and status updates exchanged whle the session runs. 


### **2. Setting up Anthropic SDK**
- Install Anthropic Python SDK from which we create agents, environments, files, and sessions. 

In [3]:
%pip install -q --upgrade anthropic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
## Loading API Key from .env file into my code
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

In [5]:
## Laoding the anthropic API key from the environment variable
import os
from anthropic import Anthropic

api_key = os.environ.get("ANTHROPIC_API_KEY")
assert api_key, "Set ANTHROPIC_API_KEY in your environment or in a local .env file."

In [6]:
## setting up the BETA version of the API
BETA_FLAG = "managed-agents-2026-04-01"

client = Anthropic(api_key=api_key)

In [7]:
## creating a dictionary to store IDs of every resource created in this notebook, so that we can clean up after ourselves
## Resources include the agent, environment, uploaded files, conversations, and the session. 

created = {"agent": None, "environment": None, "file": None, "session": None}
print("✓ Anthropic client initialized.")

✓ Anthropic client initialized.


#### **3. Creating the Managed Agent**
- A managed agent is the reusable configuration for my workflow.
- When creating a managed agent, choose Claude Model, write a system prompt defining roles and instructions, while attaching the tools and skills it can use. 
- Same agent can be reused for multiple sessions instead of recreating the configuration each time.
- I have renamed my agent to **Sonnet 5 Data Analyst** and will use the claude-sonnet-5 model.
- The system prompt tells the agent to behave like a careful data analyst: inspecting files mounted in the folder /workspace, use code execution for analysis, keep results concise, and save final files to /mnt/session/outputs.

In [8]:
agent = client.beta.agents.create(
    name="Sonnet 5 Data Analyst",
    model="claude-sonnet-5",
    system=(
        "You are a meticulous data analyst. When asked about data, always read the "
        "file mounted at /workspace, analyse it with the code execution tool, and "
        "report concise, numeric results. Use the XLSX skill for spreadsheet work. "
        "Save final artifacts to /mnt/session/outputs."
    ),
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
    skills=[{"type": "anthropic", "skill_id": "xlsx"}],
)

- The *agent_toolset_20260401* gives agent access to Anthropic's built-in tools, and the xlsx skill provides expert guidance for creating and analyzing Excel workbooks. 
- Next, we save agent ID to be used later as below. 

In [9]:
created["agent"] = agent.id
print(f"✓ Created agent: {agent.id}")

✓ Created agent: agent_01YLJEEKWdETX7L7ML53YfPb


#### **4. Configure the Anthropic Cloud Sandbox**
- We create an environment within which the Managed Agents runs during a session/task.
- An environment acts as safe and secure sandbox which gives the agent a sepparate workplace to read mounted files, write code, and run commands. 
- Our environment will be created using Anthropic's cloud environment. 

In [10]:
environment = client.beta.environments.create(
    name="code-exec-sandbox",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

created["environment"] = environment.id
print(f"✓ Created environment: {environment.id}")

✓ Created environment: env_01BSQG4JJXsY1DrMDE6Q4DnQ


#### **5. Upload Data with the Anthropic Files API**
- Next step, upload the dataset for the agent to analyze. Our **Sonnet 5 Data Analyst** uses Anthropic FIles API to be mounted inside session environment.
- After uploading the dataset, we check if file exists and confirm that it contains the expected number of data rows. 

In [14]:
from pathlib import Path

csv_path = Path("parental_leave.csv")
assert csv_path.exists(), f"Missing input file: {csv_path.resolve()}"

row_count = sum(1 for _ in csv_path.open(encoding="latin-1")) - 1
assert row_count == 1601, f"Expected 1601 data rows, found {row_count}"

- Next upload the file and store its ID for later cleanup.

In [15]:
uploaded = client.beta.files.upload(file=csv_path)

created["file"] = uploaded.id
print(f"✓ Uploaded {csv_path}: {uploaded.id} ({row_count} rows)")

✓ Uploaded parental_leave.csv: file_01UWgX8YH82z488c9g8vfhsD (1601 rows)


#### **6. Initialize an Agent Execution Session**
- Next, we create a session. A session connects the agent, the environment, and the resources it needs for a specific task.
- We then will mount the uploaded CSV file at /workspace/parental_leave.csv, so that the agent accesses it from inside the sandbox. 

In [16]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    resources=[
        {
            "type": "file",
            "file_id": uploaded.id,
            "mount_path": "/workspace/parental_leave.csv",
        },
    ],
)

created["session"] = session.id
print(f"✓ Created session: {session.id}")

✓ Created session: sesn_01WuLVwd3yHvk5qs2XQCYxpj


#### **7. Stream the Agent’s Response**
- Next step, we send task to the session and stream agent's activity as it works.
- Remember, creating a session only prepares agent and sandbox. The agent only starts working after it receives a user.message event. 
- The event stream lets us see the agent's messages, tool calls, and final session status in real time. 
- We add a prompt that tells the agent to analyze mounted CSV file, write, and run a Python Script, create a JSON summary, and building an Excel report. 

In [ ]:
agent_text_parts = []
tools_used = []
final_status = None

with client.beta.sessions.events.stream(session.id) as stream:
    # Send the user message once the stream is open.
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Use the XLSX skill and analyze /workspace/sales_data.csv. "
                            "Write /mnt/session/outputs/analyze_sales.py, run that Python "
                            "script, and have it create /mnt/session/outputs/summary.json. "
                            "Also create /mnt/session/outputs/sales_report.xlsx with the "
                            "source data, monthly profit, and a summary sheet."
                        ),
                    }
                ],
            }
        ],
    )

    for event in stream:
        etype = getattr(event, "type", None)

        if etype == "agent.message":
            for block in event.content:
                txt = getattr(block, "text", None)
                if txt:
                    print(txt, end="")
                    agent_text_parts.append(txt)

        elif etype == "agent.tool_use":
            name = getattr(event, "name", "<tool>")
            print(f"\n[tool_use] {name}")
            tools_used.append(name)

        elif etype == "session.status_idle":
            final_status = "idle"
            print("\n\n✓ Agent finished; session is idle.")
            break

        elif etype == "session.status_error":
            final_status = "error"
            print("\n✗ Session reported an error.")
            break

print("Tools used:", tools_used)